# BT4012 Competition — Supervised Learning

Continues from `preprocessing.ipynb`. We train on the 31,235 rows with a genuine known label (per that notebook's conclusion, we don't use propagated pseudo-labels) and predict on `test.csv`.

Two models are trained and compared — **Random Forest** (the strongest baseline in the original Elliptic paper, where it beat a GCN) and **XGBoost** — using a **temporal** validation split, not a random one: `test.csv` covers `time_step` 36-49, entirely after anything in `train`, so a random split would repeat the same "future leaks into past" mistake found and corrected in `preprocessing.ipynb`. `time_step` itself is excluded as a model input, since every test row's `time_step` is outside the range the model would ever see during training (tree-based models don't extrapolate beyond a feature's observed range).

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, roc_auc_score

pd.set_option("display.max_columns", 20)

## Load data and set up the labeled training set

In [2]:
train_reduced = pd.read_csv("data/train_reduced.csv")
test_reduced = pd.read_csv("data/test_reduced.csv")

# Model features: everything except identifiers/time and the label itself.
feat_cols = [c for c in train_reduced.columns if c not in ("txId", "time_step", "label")]

labeled = train_reduced[train_reduced["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

print("model features:", len(feat_cols))
print("labeled rows:", len(labeled), f"(illicit {labeled['label'].mean():.2%})")
print("test rows:", len(test_reduced))

model features: 76
labeled rows: 31235 (illicit 11.67%)
test rows: 15329


## Temporal validation split

Same split point used to validate label propagation in `preprocessing.ipynb`: train on `time_step` ≤ 30, validate on 31-35. This mirrors the real train → test gap (validation is strictly later than training) so the reported numbers reflect how the model would actually perform on genuinely future data, not an optimistic random split.

In [3]:
SPLIT_POINT = 30

train_df = labeled[labeled["time_step"] <= SPLIT_POINT]
val_df = labeled[labeled["time_step"] > SPLIT_POINT]

X_train, y_train = train_df[feat_cols], train_df["label"]
X_val, y_val = val_df[feat_cols], val_df["label"]

print(f"train: {len(train_df)} rows (illicit {y_train.mean():.2%})")
print(f"val:   {len(val_df)} rows (illicit {y_val.mean():.2%})")

train: 26905 rows (illicit 10.98%)
val:   4330 rows (illicit 15.94%)


## Model 1 — Random Forest

`class_weight="balanced"` re-weights the loss inversely to class frequency, since illicit is only ~11% of labeled rows.

In [4]:
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_val_proba = rf.predict_proba(X_val)[:, 1]
rf_val_pred = rf.predict(X_val)

print(confusion_matrix(y_val, rf_val_pred))
print(classification_report(y_val, rf_val_pred, digits=3))
print("PR-AUC (illicit):", average_precision_score(y_val, rf_val_proba))
print("ROC-AUC:", roc_auc_score(y_val, rf_val_proba))

[[3635    5]
 [ 182  508]]
              precision    recall  f1-score   support

           0      0.952     0.999     0.975      3640
           1      0.990     0.736     0.845       690

    accuracy                          0.957      4330
   macro avg      0.971     0.867     0.910      4330
weighted avg      0.958     0.957     0.954      4330

PR-AUC (illicit): 0.9893704478679826
ROC-AUC: 0.997013855709508


## Model 2 — XGBoost

`scale_pos_weight` is the standard XGBoost equivalent of `class_weight="balanced"` — the ratio of majority to minority class counts in the training fold.

In [5]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, eval_metric="logloss",
    random_state=42, n_jobs=-1,
)
xgb.fit(X_train, y_train)

xgb_val_proba = xgb.predict_proba(X_val)[:, 1]
xgb_val_pred = xgb.predict(X_val)

print(confusion_matrix(y_val, xgb_val_pred))
print(classification_report(y_val, xgb_val_pred, digits=3))
print("PR-AUC (illicit):", average_precision_score(y_val, xgb_val_proba))
print("ROC-AUC:", roc_auc_score(y_val, xgb_val_proba))

[[3614   26]
 [  14  676]]
              precision    recall  f1-score   support

           0      0.996     0.993     0.994      3640
           1      0.963     0.980     0.971       690

    accuracy                          0.991      4330
   macro avg      0.980     0.986     0.983      4330
weighted avg      0.991     0.991     0.991      4330

PR-AUC (illicit): 0.9910822078662335
ROC-AUC: 0.9980303392259914


## Comparison

| Model | Illicit precision | Illicit recall | Illicit F1 | PR-AUC |
|---|---|---|---|---|
| Random Forest | 0.990 | 0.736 | 0.845 | 0.989 |
| **XGBoost** | 0.963 | **0.980** | **0.971** | **0.991** |

**XGBoost wins clearly**, mainly on recall (0.980 vs. 0.736) — Random Forest is more conservative and misses more illicit transactions, while XGBoost catches nearly all of them at only a small precision cost. Both are dramatically better than the label-propagation approach from `preprocessing.ipynb` (illicit F1 ~0.65 at best there), confirming that a supervised model trained directly on the 76 features vastly outperforms trying to infer labels from neighbor similarity alone. **XGBoost is used for the final submission.**

## Final model — retrain on all labeled data, predict on test

The validation split above was only to compare models honestly. For the actual submission we retrain XGBoost on **all** 31,235 labeled rows (time_steps 1-35), not just the ones ≤ 30, since we want to use every known label available before predicting on the genuinely unseen `test.csv` (time_steps 36-49).

In [6]:
X_full, y_full = labeled[feat_cols], labeled["label"]
scale_pos_weight_full = (y_full == 0).sum() / (y_full == 1).sum()

final_model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_full, eval_metric="logloss",
    random_state=42, n_jobs=-1,
)
final_model.fit(X_full, y_full)

test_proba = final_model.predict_proba(test_reduced[feat_cols])[:, 1]
print("predicted illicit share on test:", (test_proba >= 0.5).mean())

predicted illicit share on test: 0.04175092961054211


## Build the submission file

`sample_submission.csv` expects `index` = the 0-based row position in `test.csv`, and `target` = the predicted probability of illicit. We already confirmed `test_reduced.csv` preserves `test.csv`'s exact row order, so a plain positional index lines up correctly.

In [7]:
submission = pd.DataFrame({
    "index": np.arange(len(test_reduced)),
    "target": test_proba,
})
submission.to_csv("data/submission.csv", index=False)

print("saved data/submission.csv:", submission.shape)
submission.head()

saved data/submission.csv: (15329, 2)


,index,target
0,0,0.000079
1,1,0.001679
2,2,0.000183
3,3,0.001047
4,4,0.000004
